In [ ]:
# Load checkpoints

MODEL_DIR = "/home/wyf/orcd/pool/reverse-llm/models"
TOKENIZER_DIR = "/home/wyf/orcd/pool/reverse-llm/tokenizers"
DATA_DIR = "/home/wyf/orcd/pool/reverse-llm/data"

model_name = "reverse-gpt2-0.35B-fineweb-10BT-ctx-1024"

from transformers import PreTrainedTokenizerFast
from transformers import GPT2LMHeadModel
from datasets import Dataset
import torch
import numpy as np
from torch.utils.data import DataLoader

import os

avail_checkpoints = sorted(os.listdir(f"{MODEL_DIR}/{model_name}"))
print("Available checkpoints:")
print("\n".join(avail_checkpoints))

In [ ]:
tensors["train"] = torch.tensor(split_datasets["train"]["input_ids"]).long()

In [ ]:
# Convert to tensor once
tensors = {
    "train": torch.tensor(split_datasets["train"]['input_ids']).long(),
    "valid": torch.tensor(split_datasets["valid"]['input_ids']).long()
}
train_dataloader = DataLoader(tensors["train"], batch_size=32, shuffle=False)
valid_dataloader = DataLoader(tensors["valid"], batch_size=32, shuffle=False)

In [ ]:
import torch
torch.cuda.empty_cache()

import gc
gc.collect()

In [ ]:
from tqdm import tqdm

def get_loss(checkpoint, split):
    model_dir = f"{MODEL_DIR}/{model_name}/{checkpoint}"
    model = GPT2LMHeadModel.from_pretrained(model_dir)
    model.to("cuda")
    model.eval()
    
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader):
            batch = batch.to("cuda")
            loss = model(batch, labels=batch).loss
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

for checkpoint in avail_checkpoints:
    print(checkpoint, "\t", get_loss(checkpoint, "train"), "\t", get_loss(checkpoint, "valid"))

In [ ]:
from transformers import pipeline

# Load tokenizer
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/fineweb_bpe_200k.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
)

# Load model
def generate_sample(checkpoint_no, input_text, **pipe_kwargs):
    model = GPT2LMHeadModel.from_pretrained(f"{MODEL_DIR}/{model_name}/checkpoint-{checkpoint_no}")

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        clean_up_tokenization_spaces=False,
        **pipe_kwargs,
    )
    text = pipe(input_text)[0]["generated_text"]
    return text

In [ ]:
print(generate_sample(9000, "is the author of Harry Potter."[::-1])[::-1])

In [ ]:
# Test if model can generate EOS tokens at all
def test_eos_generation(checkpoint_no):
    model = GPT2LMHeadModel.from_pretrained(f"{MODEL_DIR}/{model_name}/checkpoint-{checkpoint_no}")
    model.to("cuda")
    
    input_ids = tokenizer.encode("The quick brown fox"[::-1], return_tensors="pt").to("cuda")
    
    # Generate with explicit EOS stopping
    output = model.generate(
        input_ids, 
        max_new_tokens=512,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
    )
    
    new_tokens = output[0][len(input_ids[0]):]
    print(f"Generated {len(new_tokens)} tokens")
    print(f"EOS in output: {tokenizer.eos_token_id in new_tokens}")
    print(f"Stopped because: {'EOS generated' if tokenizer.eos_token_id in new_tokens else 'Hit max_new_tokens'}")
    
    return new_tokens

tokens = test_eos_generation(9000)

In [ ]:
print(tokenizer.decode(tokens[:-1])[::-1])